# imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)

In [2]:
csv_path = Path(r"C:\Users\user\Desktop\div\data\files\housing.csv")
df = pd.read_csv(csv_path)
df.head(3)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY


# outliers

### simple

In [3]:
outliers1 = df['median_house_value'] == df['median_house_value'].max()
outliers2 = df['housing_median_age'] == df['housing_median_age'].max()
outliers = outliers1 | outliers2


In [4]:
df = df.loc[~outliers, :].copy()
print(df.shape)
df.head(3)

(18572, 10)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
8,-122.26,37.84,42.0,2555.0,665.0,1206.0,595.0,2.0804,226700.0,NEAR BAY


# split

In [5]:
X = df.drop('median_house_value', axis='columns')
y = df['median_house_value']

In [6]:
X['income_cat'] = pd.cut(
    X['median_income'],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
    )

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=X['income_cat'],
    test_size=0.2
    )

X_train.shape, X_test.shape

((14857, 10), (3715, 10))

In [8]:
X_train.drop('income_cat', axis='columns', inplace=True)
X_test.drop('income_cat', axis='columns', inplace=True)

# pipeline

In [9]:
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.cluster import KMeans

n_clusters = 10
kmeans_ = KMeans(n_clusters = n_clusters)
kmeans_.fit(X_train[['latitude', 'longitude']])
print(kmeans_.cluster_centers_)

[[  37.28893588 -121.90777626]
 [  34.00644603 -118.22100414]
 [  38.34923893 -121.11836556]
 [  36.3665153  -119.48068115]
 [  40.60868571 -123.75805714]
 [  32.89988473 -116.94925793]
 [  34.82384196 -120.17277929]
 [  40.08527027 -121.79635135]
 [  34.00643885 -117.35853237]
 [  38.03728353 -122.33679117]]


In [10]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans

class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10):
        self.n_clusters = n_clusters

    def fit(self, X, y=None):
        self.kmeans_ = KMeans(self.n_clusters)
        self.kmeans_.fit(X)
        return self  

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=1).round(3)

    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]

In [11]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

cat_cols = ['ocean_proximity']
log_cols = [
    'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income'
    ]

cat_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])
log_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('log', FunctionTransformer(np.log1p, inverse_func=np.expm1, feature_names_out='one-to-one')),
    ('scl', StandardScaler())
])

def ratio_func(arr_2d):
    return arr_2d[:, [0]] / arr_2d[:, [1]]
def ratio_feature_names(*args, **kwargs):
    return ['ratio']

rat_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('newcol', FunctionTransformer(ratio_func, feature_names_out=ratio_feature_names)),
    ('scl', StandardScaler())
])
cluster_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('cluster', ClusterSimilarity(n_clusters=10))
])
num_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('scl', StandardScaler())
])

preprocessing = ColumnTransformer(
    transformers=[
        ('CAT', cat_pipeline, cat_cols),
        ('LOG', log_pipeline, log_cols),
        ('RAT_b/r', rat_pipeline, ['total_bedrooms', 'total_rooms']),
        ('RAT_r/h', rat_pipeline, ['total_rooms', 'households']),
        ('RAT_p/h', rat_pipeline, ['population', 'households']),
        ("GEO", cluster_pipeline, ["latitude", "longitude"]),
    ], remainder=num_pipeline
)

X_train_arr = preprocessing.fit_transform(X_train)
X_train_prepared = pd.DataFrame(X_train_arr, columns=preprocessing.get_feature_names_out())

X_test_arr = preprocessing.fit_transform(X_test)
X_test_prepared = pd.DataFrame(X_test_arr, columns=preprocessing.get_feature_names_out())

X_train_prepared

,CAT__ocean_proximity_<1H OCEAN,CAT__ocean_proximity_INLAND,CAT__ocean_proximity_ISLAND,CAT__ocean_proximity_NEAR BAY,CAT__ocean_proximity_NEAR OCEAN,LOG__total_rooms,LOG__total_bedrooms,LOG__population,LOG__households,LOG__median_income,RAT_b/r__ratio,RAT_r/h__ratio,RAT_p/h__ratio,GEO__Cluster 0 similarity,GEO__Cluster 1 similarity,GEO__Cluster 2 similarity,GEO__Cluster 3 similarity,GEO__Cluster 4 similarity,GEO__Cluster 5 similarity,GEO__Cluster 6 similarity,GEO__Cluster 7 similarity,GEO__Cluster 8 similarity,GEO__Cluster 9 similarity,remainder__housing_median_age
0,0.0,1.0,0.0,0.0,0.0,0.379842,0.527743,0.110166,0.493369,-1.170705,0.234906,-0.252198,-0.194074,0.000,0.619,0.069,0.003,0.000,0.000,0.000,0.306,0.000,0.055,-0.523504
1,0.0,1.0,0.0,0.0,0.0,0.129796,-0.153193,-0.024647,-0.141538,1.073846,-0.745363,0.381052,0.025317,0.000,0.470,0.001,0.022,0.000,0.000,0.000,0.762,0.000,0.622,-1.573539
2,0.0,1.0,0.0,0.0,0.0,-1.844984,-1.907327,-2.317213,-2.595613,-0.037006,-0.100671,1.354597,0.100566,0.000,0.044,0.000,0.206,0.000,0.023,0.000,0.044,0.000,0.535,-1.223527
3,1.0,0.0,0.0,0.0,0.0,0.727428,0.335237,0.385320,0.395759,1.056327,-0.995952,0.534248,-0.035513,0.800,0.000,0.000,0.000,0.240,0.000,0.009,0.000,0.750,0.000,0.264022
4,1.0,0.0,0.0,0.0,0.0,-0.791948,-0.498825,-1.486079,-0.667613,-0.384966,0.780761,-0.316516,-0.337534,0.927,0.000,0.000,0.000,0.029,0.000,0.114,0.000,0.278,0.000,1.401560
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14852,0.0,1.0,0.0,0.0,0.0,0.343598,0.500834,0.368296,0.426527,-1.489251,0.263864,-0.208652,-0.058190,0.077,0.000,0.123,0.000,0.000,0.000,0.558,0.000,0.009,0.000,1.489063
14853,1.0,0.0,0.0,0.0,0.0,0.774710,1.104456,1.218409,1.051624,-0.071093,0.711090,-0.461686,0.062246,0.962,0.000,0.000,0.000,0.115,0.000,0.028,0.000,0.560,0.000,-0.173492
14854,1.0,0.0,0.0,0.0,0.0,-0.398462,0.022781,0.687048,0.076728,-0.941885,1.145075,-0.742851,0.343559,0.999,0.000,0.000,0.000,0.067,0.000,0.050,0.000,0.510,0.000,1.139052
14855,0.0,1.0,0.0,0.0,0.0,0.608701,0.781114,0.952031,0.774793,-0.649264,0.277537,-0.317839,0.065724,0.205,0.000,0.000,0.000,0.484,0.000,0.000,0.000,0.808,0.000,0.701537


# training

In [12]:
from sklearn.linear_model import LinearRegression

lin_reg = Pipeline([
    ("cleaning", preprocessing), 
    ("linreg", LinearRegression())
    ])
lin_reg.fit(X_train, y_train)
y_pred = lin_reg.predict(X_test)

In [13]:
from sklearn.metrics import root_mean_squared_error

lin_rmse = root_mean_squared_error(y_test, y_pred)
lin_rmse

61620.826561257476

In [20]:
from sklearn.svm import SVR

svr = Pipeline([
    ("cleaning", preprocessing), 
    ("svr", SVR(kernel='rbf'))
    ])
svr.fit(X_train, y_train)
y_pred = svr.predict(X_test)
svr_rmse = root_mean_squared_error(y_test, y_pred)
svr_rmse

98475.56680978923

In [14]:
from sklearn.tree import DecisionTreeRegressor

tree_reg = Pipeline([
    ("cleaning", preprocessing), 
    ("treereg", DecisionTreeRegressor())
    ])
tree_reg.fit(X_train, y_train)
y_pred = tree_reg.predict(X_test)
tree_rmse = root_mean_squared_error(y_test, y_pred)
tree_rmse

62250.735908345305

In [21]:
from sklearn.neighbors import KNeighborsRegressor

knn_reg = Pipeline([
    ("cleaning", preprocessing), 
    ("knn_reg", KNeighborsRegressor())
    ])
knn_reg.fit(X_train, y_train)
y_pred = knn_reg.predict(X_test)
knn_rmse = root_mean_squared_error(y_test, y_pred)
knn_rmse

52709.173334909545

In [15]:
from sklearn.ensemble import RandomForestRegressor

forest_reg = Pipeline([
    ("cleaning", preprocessing), 
    ("forestreg", RandomForestRegressor())
    ])
forest_reg.fit(X_train, y_train)
y_pred = forest_reg.predict(X_test)
tree_rmse = root_mean_squared_error(y_test, y_pred)
tree_rmse

42748.169002516865

In [40]:
from sklearn.ensemble import VotingRegressor

voting_reg = Pipeline([
    ("cleaning", preprocessing), 
    ("forestreg", VotingRegressor([
		('lr', KNeighborsRegressor()),
		('rf', RandomForestRegressor(random_state=123))
    	])
	)
])
voting_reg.fit(X_train, y_train)
y_pred = voting_reg.predict(X_test)
voting_rmse = root_mean_squared_error(y_test, y_pred)
voting_rmse

44936.85200428093

In [35]:
from sklearn.ensemble import GradientBoostingRegressor

gb_reg = forest_reg = Pipeline([
    ("cleaning", preprocessing), 
    ("forestreg", GradientBoostingRegressor(
        n_estimators=10,
        max_depth=10,)
    )
])
gb_reg.fit(X_train, y_train)
y_pred = gb_reg.predict(X_test)
gb_rmse = root_mean_squared_error(y_test, y_pred)
gb_rmse

54985.689595321215

In [37]:
from sklearn.ensemble import BaggingRegressor

bag_reg = Pipeline([
    ("cleaning", preprocessing), 
    ("knn_reg", BaggingRegressor(
		KNeighborsRegressor(3), 
		n_estimators=500,
		max_samples=len(X_train) // 4, 
		bootstrap=True,
		n_jobs=-1, 
		random_state=123)
    )]
)
bag_reg.fit(X_train, y_train)
y_pred = bag_reg.predict(X_test)
bag_rmse = root_mean_squared_error(y_test, y_pred)
bag_rmse

51975.41076034463

In [38]:
from sklearn.ensemble import AdaBoostRegressor

adab_reg = Pipeline([
    ("cleaning", preprocessing), 
    ("adab_reg", AdaBoostRegressor(
		KNeighborsRegressor(3), 
		n_estimators=100,
		random_state=123)
    )]
)
adab_reg.fit(X_train, y_train)
y_pred = adab_reg.predict(X_test)
adab_rmse = root_mean_squared_error(y_test, y_pred)
adab_rmse

64060.39966929959

In [16]:
# import joblib
# joblib.dump(forest_reg, "housing_random_forest_full.pkl")

In [17]:
# full_pipeline = joblib.load('housing_random_forest_full.pkl')
# full_pipeline.predict(X_test[0:1])

# cross-validation

In [ ]:
from sklearn.model_selection import cross_val_score

forest_rmses = -cross_val_score(
    forest_reg, 
    X.drop('income_cat', axis='columns', errors='ignore'), 
    y,
	scoring="neg_root_mean_squared_error", 
    cv=10)
forest_rmses, forest_rmses.mean()

(array([44537.67035673, 43092.32307376, 44223.10222122, 42715.92379338,
        40329.19699297, 40254.93523975, 40429.11832713, 40736.72680285,
        43723.37563021, 42057.62779334]),
 np.float64(42210.00002313345))

In [41]:
voting_rmses = -cross_val_score(
    voting_reg, 
    X.drop('income_cat', axis='columns', errors='ignore'), 
    y,
	scoring="neg_root_mean_squared_error", 
    cv=10)
voting_rmses, voting_rmses.mean()

(array([46240.23369409, 44833.69533805, 46086.78275242, 45575.49250751,
        43122.9909236 , 43864.5721931 , 43500.71943428, 42839.20313753,
        46304.07066701, 44874.21266784]),
 np.float64(44724.19733154322))

# Fine-Tune Your Model

## Grid Search

### random forest

In [ ]:
from sklearn.model_selection import GridSearchCV

full_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("random_forest", RandomForestRegressor(random_state=42)),
])
param_grid = [
    {'preprocessing__GEO__cluster__n_clusters': [5, 8, 10],
     'random_forest__max_features': [4, 6, 8]},
    {'preprocessing__GEO__cluster__n_clusters': [10, 15],
     'random_forest__max_features': [6, 8, 10]},
]
grid_search = GridSearchCV(
    full_pipeline, 
    param_grid, 
    cv=3, # 3 folds
    scoring='neg_root_mean_squared_error'
)
grid_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'preprocessing__GEO__cluster__n_clusters': [5, 8, ...], 'random_forest__max_features': [4, 6, ...]}, {'preprocessing__GEO__cluster__n_clusters': [10, 15], 'random_forest__max_features': [6, 8, ...]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intCont

- Grid 1: Tests 3 cluster sizes × 3 feature counts = 9 combinations.
- Grid 2: Tests 2 cluster sizes × 3 feature counts = 6 combinations.
- Total: 15 unique model configurations.

In [24]:
grid_search.best_params_

{'preprocessing__GEO__cluster__n_clusters': 15,
 'random_forest__max_features': 6}

In [26]:
cv_res = pd.DataFrame(grid_search.cv_results_)
cv_res.sort_values(by="mean_test_score", ascending=False, inplace=True)

cv_res = cv_res[[
    "param_preprocessing__GEO__cluster__n_clusters",
    "param_random_forest__max_features", 
    "split0_test_score",
    "split1_test_score", 
    "split2_test_score", 
    "mean_test_score"]]
score_cols = ["split0", "split1", "split2", "mean_test_rmse"]
cv_res.columns = ["n_clusters", "max_features"] + score_cols
cv_res[score_cols] = -cv_res[score_cols].round().astype(np.int64)

cv_res.head()

,n_clusters,max_features,split0,split1,split2,mean_test_rmse
12,15,6,41843,40045,39695,40528
13,15,8,41940,40112,39888,40647
14,15,10,41895,40368,40191,40818
7,10,6,42762,41216,41207,41728
9,10,6,43009,41255,41004,41756


### voting

In [42]:
from sklearn.model_selection import GridSearchCV

full_pipeline = Pipeline([
    ("preprocessing", preprocessing), 
    ("votingreg", VotingRegressor([
		('knn', KNeighborsRegressor()),
		('rf', RandomForestRegressor(random_state=123))
    	])
	)
])
param_grid = [
    {'preprocessing__GEO__cluster__n_clusters': [10, 15],
     'votingreg__knn__n_neighbors': [3, 5, 10],
     'votingreg__rf__max_features': [4, 6, 8]},
    # {},
]
grid_search = GridSearchCV(
    full_pipeline, 
    param_grid, 
    cv=3, # 3 folds
    scoring='neg_root_mean_squared_error'
)
grid_search.fit(
    X_train, 
    y_train
)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...ate=123))]))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'preprocessing__GEO__cluster__n_clusters': [10, 15], 'votingreg__knn__n_neighbors': [3, 5, ...], 'votingreg__rf__max_features': [4, 6, ...]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : t

In [43]:
grid_search.best_params_

{'preprocessing__GEO__cluster__n_clusters': 15,
 'votingreg__knn__n_neighbors': 5,
 'votingreg__rf__max_features': 8}

In [ ]:
cv_res = pd.DataFrame(grid_search.cv_results_)
cv_res.sort_values(by="mean_test_score", ascending=False, inplace=True)

cv_res.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_preprocessing__GEO__cluster__n_clusters,param_votingreg__knn__n_neighbors,param_votingreg__rf__max_features,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score
14,3.112353,0.162209,0.220120,0.003792,15,5,8,{'preprocessing__GEO__cluster__n_clusters': 15...,-45660.572034,-42887.251393,-45432.886653,-44660.236693,1257.131076,1
13,2.425038,0.083160,0.216849,0.001002,15,5,6,{'preprocessing__GEO__cluster__n_clusters': 15...,-45746.871139,-43067.798388,-45375.184741,-44729.951423,1185.074464,2
16,2.421184,0.025643,0.232545,0.004951,15,10,6,{'preprocessing__GEO__cluster__n_clusters': 15...,-46149.590960,-43018.151744,-45260.715413,-44809.486039,1317.620039,3
17,3.071791,0.054750,0.231455,0.002553,15,10,8,{'preprocessing__GEO__cluster__n_clusters': 15...,-46087.913001,-42992.711134,-45388.528662,-44823.050932,1325.365973,4
12,1.750409,0.009737,0.215943,0.002501,15,5,4,{'preprocessing__GEO__cluster__n_clusters': 15...,-46255.975644,-43175.614486,-45717.759415,-45049.783182,1343.329298,5


## Randomized Search

In [28]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_distribs = {
    'preprocessing__GEO__cluster__n_clusters': randint(low=3, high=50),
    'random_forest__max_features': randint(low=2, high=20)
}

rnd_search = RandomizedSearchCV(
    full_pipeline, 
    param_distributions=param_distribs, 
    n_iter=10, 
    cv=3,
    scoring='neg_root_mean_squared_error', 
    random_state=123)

rnd_search.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'preprocessing__GEO__cluster__n_clusters': <scipy.stats....0016C6E8E1E90>, 'random_forest__max_features': <scipy.stats....0016C6E8E1550>}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation stra

In [29]:
cv_res = pd.DataFrame(rnd_search.cv_results_)
cv_res.sort_values(by="mean_test_score", ascending=False, inplace=True)
cv_res = cv_res[[
    "param_preprocessing__GEO__cluster__n_clusters",
    "param_random_forest__max_features", 
    "split0_test_score",
    "split1_test_score", 
    "split2_test_score", 
    "mean_test_score"
]]
cv_res.columns = ["n_clusters", "max_features"] + score_cols
cv_res[score_cols] = -cv_res[score_cols].round().astype(np.int64)
cv_res.head()

,n_clusters,max_features,split0,split1,split2,mean_test_rmse
0,48,4,40147,38974,38162,39094
1,31,4,40452,38838,38726,39338
2,41,19,40859,39249,38767,39625
5,35,19,41056,39510,39443,40003
4,25,3,41149,39780,39158,40029


## Evaluate Your System on the Test Set

In [30]:
final_model = rnd_search.best_estimator_

final_predictions = final_model.predict(X_test)

final_rmse = root_mean_squared_error(y_test, final_predictions)
print(final_rmse)

37806.36784289534


In [ ]:
# import joblib
# joblib.dump(forest_reg, "california_housing_final_model.pkl")

In [ ]:
# final_model_reloaded = joblib.load("california_housing_final_model.pkl")

# new_data = X_test.iloc[:5]  
# predictions = final_model_reloaded.predict(new_data)
# predictions